# HDBSCAN: embedding-model × reducer × component sweep

This notebook compares five embedding variants for clustering TikZ images:

1. **CLIP**
2. **SigLIP2**
3. **DINOv2**
4. **CLIP + DINOv2**
5. **SigLIP2 + DINOv2**

For every embedding variant, the notebook automatically tests both:

- **PCA**
- **UMAP**

with multiple target dimensions, followed by HDBSCAN.

For every run it reports:

- reducer
- target dimensions
- explained PCA variance where applicable
- number of discovered clusters
- noise fraction
- Silhouette Score on non-noise samples


In [1]:
from pathlib import Path
import gc

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from datasets import load_dataset, concatenate_datasets
from transformers import AutoModel, AutoProcessor
from sklearn.decomposition import PCA
from sklearn.cluster import HDBSCAN
from sklearn.metrics import silhouette_score
from umap import UMAP
from tqdm.auto import tqdm


/home/jonas/miniconda3/envs/data-tools/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Configuration

In [2]:
# Dataset paths
DATIKZ_DIR = Path("/home/jonas/Datasets/TikZ/DatikZ-v4-raw")
BENCHMARK_DIR = Path("/home/jonas/Datasets/TikZ/benchmark-raw")

# Choose one or both datasets:
# ("datikz",)
# ("benchmark",)
# ("datikz", "benchmark")
DATASETS_TO_USE = ("datikz", "benchmark")

# Input schemas
DATIKZ_IMAGE_COL = "png_image"
DATIKZ_ID_COL = "file_id"
DATIKZ_SOURCE_COL = "source"

BENCHMARK_IMAGE_COL = "image"
BENCHMARK_ID_COL = "uri"
BENCHMARK_SOURCE_COL = "origin"

# None uses the complete dataset. Sampling is applied per dataset.
N_SAMPLES_DATIKZ = None
N_SAMPLES_BENCHMARK = None

IMAGE_COL = "image"
BATCH_SIZE = 128
SEED = 42

# Base embedding models.
MODEL_IDS = {
    "clip": "openai/clip-vit-base-patch32",
    "siglip2": "google/siglip2-base-patch16-224",
    "dinov2": "facebook/dinov2-base",
}

EMBEDDING_VARIANTS = [
    "clip",
    "siglip2",
    "dinov2",
    "clip+dinov2",
    "siglip2+dinov2",
]

# Tested for BOTH PCA and UMAP.
COMPONENTS_TO_TEST = [
    5,
    10,
    15,
    20,
    25,
    30,
    35,
    40,
    80,
    128,
]

REDUCERS = [
    "pca",
    "umap",
]

# UMAP
UMAP_N_NEIGHBORS = 30
UMAP_MIN_DIST = 0.0
UMAP_METRIC = "cosine"

# HDBSCAN
MIN_CLUSTER_SIZE = 200
MIN_SAMPLES = 20
CLUSTER_SELECTION_METHOD = "eom"
CLUSTER_SELECTION_EPSILON = 0.0

# Silhouette is evaluated on at most this many non-noise samples.
SILHOUETTE_SAMPLE_SIZE = 5_000

# Selection used by the later visualization/export cells.
# True: use the valid run with the highest Silhouette Score.
# False: use the manually configured run below.
AUTO_SELECT_BEST_RUN = False

FINAL_EMBEDDING_VARIANT = "clip"
FINAL_REDUCER = "umap"
FINAL_COMPONENTS = 20

OUTPUT_PREFIX = "hdbscan_embedding_reducer_sweep_image"


## 2. Load data

In [3]:
def load_local_dataset(path: Path):
    parquet_files = sorted(path.rglob("*.parquet"))

    if not parquet_files:
        raise FileNotFoundError(
            f"No Parquet files found in {path}"
        )

    return load_dataset(
        "parquet",
        data_files=[str(path) for path in parquet_files],
        split="train",
    )


def normalize_dataset(
    dataset,
    dataset_name: str,
    image_col: str,
    id_col: str,
    source_col: str,
):
    required = {image_col, id_col, source_col}
    missing = required - set(dataset.column_names)

    if missing:
        raise ValueError(
            f"{dataset_name} is missing columns: "
            f"{sorted(missing)}"
        )

    if image_col != IMAGE_COL:
        dataset = dataset.rename_column(
            image_col,
            IMAGE_COL,
        )

    has_caption = "caption" in dataset.column_names

    def add_metadata(row, index):
        raw_id = row[id_col]

        return {
            "dataset": dataset_name,
            "sample_id": (
                f"{dataset_name}:"
                f"{raw_id if raw_id is not None else index}"
            ),
            "source": row[source_col],
            "caption": (
                row["caption"]
                if has_caption
                else ""
            ),
        }

    dataset = dataset.map(
        add_metadata,
        with_indices=True,
        desc=f"Normalize {dataset_name}",
    )

    return dataset.select_columns(
        [
            IMAGE_COL,
            "dataset",
            "sample_id",
            "source",
            "caption",
        ]
    )


def sample_dataset(dataset, n_samples, seed):
    if n_samples is None:
        return dataset

    n = min(int(n_samples), len(dataset))

    return (
        dataset
        .shuffle(seed=seed)
        .select(range(n))
    )


selected_datasets = set(DATASETS_TO_USE)
allowed_datasets = {"datikz", "benchmark"}

if not selected_datasets:
    raise ValueError(
        "DATASETS_TO_USE must not be empty."
    )

unknown = selected_datasets - allowed_datasets

if unknown:
    raise ValueError(
        f"Unknown dataset names: {sorted(unknown)}"
    )

datasets_to_cluster = []

if "datikz" in selected_datasets:
    datikz = normalize_dataset(
        load_local_dataset(DATIKZ_DIR),
        dataset_name="datikz",
        image_col=DATIKZ_IMAGE_COL,
        id_col=DATIKZ_ID_COL,
        source_col=DATIKZ_SOURCE_COL,
    )
    datikz = sample_dataset(
        datikz,
        N_SAMPLES_DATIKZ,
        SEED,
    )
    datasets_to_cluster.append(datikz)

if "benchmark" in selected_datasets:
    benchmark = normalize_dataset(
        load_local_dataset(BENCHMARK_DIR),
        dataset_name="benchmark",
        image_col=BENCHMARK_IMAGE_COL,
        id_col=BENCHMARK_ID_COL,
        source_col=BENCHMARK_SOURCE_COL,
    )
    benchmark = sample_dataset(
        benchmark,
        N_SAMPLES_BENCHMARK,
        SEED + 1,
    )
    datasets_to_cluster.append(benchmark)

sample = concatenate_datasets(
    datasets_to_cluster
).shuffle(seed=SEED)

print(sample)
print()
print(
    pd.Series(sample["dataset"])
    .value_counts()
)


Generating train split: 427753 examples [00:17, 23853.41 examples/s]
Normalize datikz: 100%|██████████| 427753/427753 [00:40<00:00, 10647.43 examples/s]
Generating train split: 167694 examples [00:23, 7062.69 examples/s] 
Normalize benchmark: 100%|██████████| 167694/167694 [00:30<00:00, 5554.31 examples/s]


Dataset({
    features: ['image', 'dataset', 'sample_id', 'source', 'caption'],
    num_rows: 595447
})

datikz       427753
benchmark    167694
Name: count, dtype: int64


In [4]:
print(f"Used images: {len(sample):,}")

if len(sample) < 3:
    raise ValueError(
        "At least three samples are required."
    )


Used images: 595,447


## 3. Compute base image embeddings

Only the three base models are run:

- CLIP
- SigLIP2
- DINOv2

The combined variants are generated afterwards without another model forward pass.


In [5]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

for name, model_id in MODEL_IDS.items():
    print(f"{name:8s} -> {model_id}")


Device: cuda
clip     -> openai/clip-vit-base-patch32
siglip2  -> google/siglip2-base-patch16-224
dinov2   -> facebook/dinov2-base


In [6]:
def encode_base_model(
    model_name: str,
    model_id: str,
) -> np.ndarray:
    processor = AutoProcessor.from_pretrained(
        model_id
    )
    model = (
        AutoModel
        .from_pretrained(model_id)
        .eval()
        .to(device)
    )

    all_embeddings = []

    for start in tqdm(
        range(0, len(sample), BATCH_SIZE),
        desc=f"Encode {model_name}",
    ):
        batch = sample[
            start : start + BATCH_SIZE
        ]

        images = [
            image.convert("RGB")
            for image in batch[IMAGE_COL]
        ]

        pixel_values = processor(
            images=images,
            return_tensors="pt",
        )["pixel_values"].to(device)

        with torch.inference_mode():
            if model_name in {
                "clip",
                "siglip2",
            }:
                vision_output = model.vision_model(
                    pixel_values=pixel_values,
                    return_dict=True,
                )

                features = (
                    vision_output.pooler_output
                )

                projection = getattr(
                    model,
                    "visual_projection",
                    None,
                )

                if projection is not None:
                    features = projection(
                        features
                    )

            elif model_name == "dinov2":
                output = model(
                    pixel_values=pixel_values,
                    return_dict=True,
                )

                features = (
                    output.last_hidden_state[
                        :, 0
                    ]
                )

            else:
                raise ValueError(
                    f"Unknown model: {model_name}"
                )

            if not isinstance(
                features,
                torch.Tensor,
            ):
                raise TypeError(
                    "Expected embedding tensor, "
                    f"got {type(features).__name__}"
                )

            features = F.normalize(
                features.to(torch.float32),
                dim=-1,
            )

        all_embeddings.append(
            features.cpu().numpy()
        )

    embeddings = np.concatenate(
        all_embeddings,
        axis=0,
    )

    del model
    del processor
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(
        f"{model_name}: "
        f"{embeddings.shape}"
    )

    return embeddings


base_embeddings = {}

for model_name, model_id in MODEL_IDS.items():
    base_embeddings[model_name] = (
        encode_base_model(
            model_name,
            model_id,
        )
    )

    np.save(
        f"{OUTPUT_PREFIX}_{model_name}_embeddings.npy",
        base_embeddings[model_name],
    )


Encode clip: 100%|██████████| 4652/4652 [34:18<00:00,  2.26it/s]


clip: (595447, 512)


[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.
Encode siglip2: 100%|██████████| 4652/4652 [35:13<00:00,  2.20it/s]


siglip2: (595447, 768)


Encode dinov2: 100%|██████████| 4652/4652 [42:54<00:00,  1.81it/s]


dinov2: (595447, 768)


## 4. Build all embedding variants

For combined variants, each base embedding is L2-normalized before concatenation. The concatenated vector is normalized again.

This gives both encoders comparable contribution regardless of their original embedding dimensionality.


In [7]:
def normalize_numpy(
    embeddings: np.ndarray,
) -> np.ndarray:
    norms = np.linalg.norm(
        embeddings,
        axis=1,
        keepdims=True,
    )

    return embeddings / np.maximum(
        norms,
        1e-12,
    )


def combine_embeddings(
    first: np.ndarray,
    second: np.ndarray,
) -> np.ndarray:
    combined = np.concatenate(
        [
            normalize_numpy(first),
            normalize_numpy(second),
        ],
        axis=1,
    )

    return normalize_numpy(combined)


embedding_variants = {
    "clip": base_embeddings["clip"],
    "siglip2": base_embeddings["siglip2"],
    "dinov2": base_embeddings["dinov2"],
    "clip+dinov2": combine_embeddings(
        base_embeddings["clip"],
        base_embeddings["dinov2"],
    ),
    "siglip2+dinov2": combine_embeddings(
        base_embeddings["siglip2"],
        base_embeddings["dinov2"],
    ),
}

for name in EMBEDDING_VARIANTS:
    print(
        f"{name:18s}",
        embedding_variants[name].shape,
    )


clip               (595447, 512)
siglip2            (595447, 768)
dinov2             (595447, 768)
clip+dinov2        (595447, 1280)
siglip2+dinov2     (595447, 1536)


## 5. Automatic embedding × reducer × component × HDBSCAN sweep

Every embedding variant is tested with both PCA and UMAP.

For PCA, the model is fitted once per embedding variant using the largest
requested number of components. Smaller PCA representations reuse the first
principal components.

UMAP must be fitted separately for every requested target dimension.

HDBSCAN is fitted separately on every reduced representation.


In [8]:
def evaluate_hdbscan(X: np.ndarray) -> dict:
    clusterer = HDBSCAN(
        min_cluster_size=MIN_CLUSTER_SIZE,
        min_samples=MIN_SAMPLES,
        cluster_selection_method=CLUSTER_SELECTION_METHOD,
        cluster_selection_epsilon=CLUSTER_SELECTION_EPSILON,
        metric="euclidean",
        n_jobs=-1,
    )

    labels = clusterer.fit_predict(X)
    mask = labels >= 0
    clustered_labels = labels[mask]
    clustered_X = X[mask]

    n_clusters = len(np.unique(clustered_labels))
    noise_fraction = float(np.mean(labels == -1))
    silhouette = np.nan

    if n_clusters >= 2 and len(clustered_X) > n_clusters:
        silhouette = silhouette_score(
            clustered_X,
            clustered_labels,
            metric="euclidean",
            sample_size=min(
                SILHOUETTE_SAMPLE_SIZE,
                len(clustered_X),
            ),
            random_state=SEED,
        )

    return {
        "n_clusters": n_clusters,
        "noise_fraction": noise_fraction,
        "silhouette": silhouette,
    }


sweep_rows = []

for embedding_name in EMBEDDING_VARIANTS:
    E = embedding_variants[embedding_name]

    # PCA: fit once, then slice.
    if "pca" in REDUCERS:
        max_components = min(
            max(COMPONENTS_TO_TEST),
            E.shape[0] - 1,
            E.shape[1],
        )

        pca = PCA(
            n_components=max_components,
            random_state=SEED,
        )
        X_full = pca.fit_transform(E)

        for requested_components in tqdm(
            COMPONENTS_TO_TEST,
            desc=f"PCA + HDBSCAN: {embedding_name}",
        ):
            n_components = min(
                requested_components,
                X_full.shape[1],
            )
            X = X_full[:, :n_components]
            metrics = evaluate_hdbscan(X)

            sweep_rows.append({
                "embedding": embedding_name,
                "reducer": "pca",
                "components": n_components,
                "pca_variance": float(
                    pca.explained_variance_ratio_[
                        :n_components
                    ].sum()
                ),
                **metrics,
            })

        del X_full, pca
        gc.collect()

    # UMAP: fit separately for every target dimension.
    if "umap" in REDUCERS:
        for requested_components in tqdm(
            COMPONENTS_TO_TEST,
            desc=f"UMAP + HDBSCAN: {embedding_name}",
        ):
            n_components = min(
                requested_components,
                E.shape[0] - 2,
            )

            reducer = UMAP(
                n_components=n_components,
                n_neighbors=UMAP_N_NEIGHBORS,
                min_dist=UMAP_MIN_DIST,
                metric=UMAP_METRIC,
                random_state=SEED,
            )
            X = reducer.fit_transform(E)
            metrics = evaluate_hdbscan(X)

            sweep_rows.append({
                "embedding": embedding_name,
                "reducer": "umap",
                "components": n_components,
                "pca_variance": np.nan,
                **metrics,
            })

            del X, reducer
            gc.collect()


sweep_results = (
    pd.DataFrame(sweep_rows)
    .sort_values(
        ["embedding", "reducer", "components"]
    )
    .reset_index(drop=True)
)

display(sweep_results)

sweep_results.to_csv(
    f"{OUTPUT_PREFIX}_metrics.csv",
    index=False,
)


PCA + HDBSCAN: clip:   0%|          | 0/10 [00:00<?, ?it/s]/home/jonas/miniconda3/envs/data-tools/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
PCA + HDBSCAN: clip:  10%|█         | 1/10 [19:18<2:53:49, 1158.82s/it]/home/jonas/miniconda3/envs/data-tools/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
PCA + HDBSCAN: clip:  10%|█         | 1/10 [43:47<6:34:04, 2627.22s/it]


KeyboardInterrupt: 

## 6. Ranking

All valid embedding/reducer/component combinations are ranked by Silhouette
Score and then by lower noise fraction.

For UMAP, `pca_variance` is empty because UMAP has no explained-variance
measure comparable to PCA.

A high Silhouette Score should always be considered together with:

- noise fraction
- number of clusters
- visual coherence of the clusters


In [ ]:
ranking = (
    sweep_results
    .dropna(subset=["silhouette"])
    .sort_values(
        [
            "silhouette",
            "noise_fraction",
        ],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

display(ranking)


## 7. Comparison plots

Each line represents one embedding + reducer combination.


In [ ]:
def plot_metric(metric, ylabel, title):
    plt.figure(figsize=(11, 5))

    for embedding_name in EMBEDDING_VARIANTS:
        for reducer_name in REDUCERS:
            subset = sweep_results[
                (sweep_results["embedding"] == embedding_name)
                & (sweep_results["reducer"] == reducer_name)
            ]

            plt.plot(
                subset["components"],
                subset[metric],
                marker="o",
                label=f"{embedding_name} + {reducer_name}",
            )

    plt.xlabel("Reduced dimensions")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend(fontsize=8)
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()


plot_metric(
    "silhouette",
    "Silhouette Score",
    "Silhouette Score by embedding, reducer and dimension",
)

plot_metric(
    "noise_fraction",
    "Noise fraction",
    "HDBSCAN noise fraction",
)

plot_metric(
    "n_clusters",
    "Clusters found",
    "Number of HDBSCAN clusters",
)


# Explained variance exists only for PCA.
plt.figure(figsize=(10, 5))

for embedding_name in EMBEDDING_VARIANTS:
    subset = sweep_results[
        (sweep_results["embedding"] == embedding_name)
        & (sweep_results["reducer"] == "pca")
    ]

    plt.plot(
        subset["components"],
        subset["pca_variance"],
        marker="o",
        label=embedding_name,
    )

plt.xlabel("PCA components")
plt.ylabel("Explained variance")
plt.title("Explained PCA variance")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## 8. Select the final run

With `AUTO_SELECT_BEST_RUN = True`, the valid combination with the highest
Silhouette Score is selected automatically.

With `AUTO_SELECT_BEST_RUN = False`, use:

- `FINAL_EMBEDDING_VARIANT`
- `FINAL_REDUCER`
- `FINAL_COMPONENTS`

The selected reducer and HDBSCAN model are fitted again for the later cells.


In [ ]:
if AUTO_SELECT_BEST_RUN:
    if ranking.empty:
        raise ValueError(
            "No valid run produced at least two non-noise clusters."
        )

    selected_embedding_name = str(
        ranking.iloc[0]["embedding"]
    )
    selected_reducer_name = str(
        ranking.iloc[0]["reducer"]
    )
    selected_components = int(
        ranking.iloc[0]["components"]
    )
else:
    selected_embedding_name = FINAL_EMBEDDING_VARIANT
    selected_reducer_name = FINAL_REDUCER
    selected_components = FINAL_COMPONENTS


E = embedding_variants[selected_embedding_name]

if selected_reducer_name == "pca":
    n_components = min(
        selected_components,
        E.shape[0] - 1,
        E.shape[1],
    )

    reducer = PCA(
        n_components=n_components,
        random_state=SEED,
    )
    X = reducer.fit_transform(E)
    reducer_info = (
        f"variance="
        f"{reducer.explained_variance_ratio_.sum():.3f}"
    )

elif selected_reducer_name == "umap":
    n_components = min(
        selected_components,
        E.shape[0] - 2,
    )

    reducer = UMAP(
        n_components=n_components,
        n_neighbors=UMAP_N_NEIGHBORS,
        min_dist=UMAP_MIN_DIST,
        metric=UMAP_METRIC,
        random_state=SEED,
    )
    X = reducer.fit_transform(E)
    reducer_info = ""

else:
    raise ValueError(
        f"Unknown reducer: {selected_reducer_name}"
    )


clusterer = HDBSCAN(
    min_cluster_size=MIN_CLUSTER_SIZE,
    min_samples=MIN_SAMPLES,
    cluster_selection_method=CLUSTER_SELECTION_METHOD,
    cluster_selection_epsilon=CLUSTER_SELECTION_EPSILON,
    metric="euclidean",
    n_jobs=-1,
)

labels = clusterer.fit_predict(X)
probabilities = clusterer.probabilities_

cluster_ids = sorted(
    int(label)
    for label in np.unique(labels)
    if label >= 0
)

n_clusters = len(cluster_ids)
n_noise = int(np.sum(labels == -1))

print("Embedding:", selected_embedding_name)
print("Reducer:", selected_reducer_name)
print("Components:", n_components)

if reducer_info:
    print(reducer_info)

print("Clusters found:", n_clusters)
print(
    "Noise:",
    f"{n_noise:,} ({n_noise / len(labels):.2%})",
)


## 9. Distribution of the selected HDBSCAN clustering

In [ ]:
cluster_labels = np.where(
    labels == -1,
    "noise",
    np.char.add(
        "cluster_",
        labels.astype(str),
    ),
)

cluster_counts = (
    pd.DataFrame(
        {
            "dataset": sample["dataset"],
            "cluster": cluster_labels,
        }
    )
    .groupby(
        ["cluster", "dataset"]
    )
    .size()
    .unstack(fill_value=0)
)

cluster_counts["total"] = (
    cluster_counts.sum(axis=1)
)

cluster_counts["percent_total"] = (
    100
    * cluster_counts["total"]
    / len(labels)
)

display(cluster_counts)

cluster_counts.drop(
    columns="percent_total"
).plot.bar(
    figsize=(11, 5),
    title=(
        f"{selected_embedding_name} | "
        f"{selected_reducer_name}={n_components}"
    ),
    xlabel="Cluster",
    ylabel="Count",
)

plt.tight_layout()
plt.show()


## 10. Two-dimensional overview

This is only a visualization of the selected reduced representation `X`.
HDBSCAN itself operates on all selected reduced dimensions.


In [ ]:
xy = PCA(
    n_components=2,
    random_state=SEED,
).fit_transform(X)

plt.figure(figsize=(9, 7))

plt.scatter(
    xy[:, 0],
    xy[:, 1],
    c=labels,
    s=8,
    alpha=0.65,
)

plt.title(
    f"{selected_embedding_name} + "
    f"{selected_reducer_name.upper()} + HDBSCAN | "
    f"{n_components} dimensions"
)
plt.xlabel("Projection 1")
plt.ylabel("Projection 2")
plt.tight_layout()
plt.show()


## 11. Representative images per cluster

HDBSCAN has no K-Means centroid. Images with the highest HDBSCAN membership probability are shown first.

Noise can be displayed separately.


In [ ]:
def show_cluster(
    cluster_id: int,
    n: int = 20,
):
    indices = np.where(
        labels == cluster_id
    )[0]

    if len(indices) == 0:
        print(
            f"Cluster {cluster_id} is empty."
        )
        return

    if cluster_id == -1:
        selected = indices[:n]
        name = "Noise"
    else:
        strengths = probabilities[indices]

        selected = indices[
            np.argsort(strengths)[::-1][:n]
        ]

        name = f"Cluster {cluster_id}"

    cols = min(5, len(selected))
    rows = int(
        np.ceil(len(selected) / cols)
    )

    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=(
            3.0 * cols,
            3.3 * rows,
        ),
        squeeze=False,
    )

    axes = axes.ravel()

    for axis in axes:
        axis.axis("off")

    for axis, index in zip(
        axes,
        selected,
    ):
        row = sample[int(index)]

        axis.imshow(
            row[IMAGE_COL]
        )

        title = (
            f"{row['dataset']} | "
            f"{index}"
        )

        if cluster_id != -1:
            title += (
                f" | p="
                f"{probabilities[index]:.2f}"
            )

        axis.set_title(
            title,
            fontsize=8,
        )

    fig.suptitle(
        f"{name} — "
        f"{len(indices):,} images",
        fontsize=14,
    )

    plt.tight_layout()
    plt.show()


for cluster_id in cluster_ids:
    show_cluster(
        cluster_id,
        n=20,
    )

if np.any(labels == -1):
    show_cluster(
        -1,
        n=20,
    )


## 12. Export selected cluster assignments

In [ ]:
results = pd.DataFrame(
    {
        "sample_index": np.arange(len(sample)),
        "dataset": sample["dataset"],
        "sample_id": sample["sample_id"],
        "source": sample["source"],
        "cluster": labels,
        "membership_probability": probabilities,
    }
)

results["class_name"] = np.where(
    results["cluster"] == -1,
    "noise",
    "class_" + (results["cluster"] + 1).astype(str),
)

display(results.head())

display(
    pd.crosstab(
        results["class_name"],
        results["dataset"],
        margins=True,
    )
)

safe_embedding_name = selected_embedding_name.replace(
    "+",
    "_plus_",
)

output_file = (
    f"{OUTPUT_PREFIX}_"
    f"{safe_embedding_name}_"
    f"{selected_reducer_name}{n_components}_"
    "assignments.csv"
)

results.to_csv(
    output_file,
    index=False,
)

print("Saved:", output_file)


## 13. Final Silhouette Score

Noise samples are excluded from the score.


In [ ]:
mask = labels >= 0
clustered_X = X[mask]
clustered_labels = labels[mask]

if len(np.unique(clustered_labels)) < 2:
    print(
        "Silhouette Score is not defined: "
        "fewer than two non-noise clusters."
    )
else:
    score = silhouette_score(
        clustered_X,
        clustered_labels,
        metric="euclidean",
        sample_size=min(
            SILHOUETTE_SAMPLE_SIZE,
            len(clustered_X),
        ),
        random_state=SEED,
    )

    print("Embedding:", selected_embedding_name)
    print("Reducer:", selected_reducer_name)
    print("Components:", n_components)
    print("Clusters:", n_clusters)
    print(
        "Noise fraction:",
        f"{np.mean(labels == -1):.2%}",
    )
    print(
        "Silhouette Score:",
        round(score, 4),
    )
